In [2]:
import cv2
import numpy as np
import os

video_path = "video_AI_Project_ENG51_1705.mp4"
if not os.path.exists(video_path):
    video_path = "cctv_demo.mp4"

cap = cv2.VideoCapture(video_path)
cap.set(cv2.CAP_PROP_POS_FRAMES, 30)
ret, orig_frame = cap.read()
cap.release()

if not ret:
    print("❌ ไม่พบไฟล์วิดีโอ")
    exit()

base_frame = cv2.resize(orig_frame, (1280, 720))
H, W = base_frame.shape[:2]

# ตัวแปรควบคุมการซูมและการเลื่อน
zoom_level = 1.0
pan_x, pan_y = 0, 0
is_dragging = False
drag_start = (0, 0)

points_real = []  # เก็บพิกัดภาพจริง (1280x720)
all_slots = []

def to_screen_coords(rx, ry):
    """แปลงพิกัดจริง -> พิกัดบนหน้าจอที่ซูม"""
    sx = int((rx - pan_x) * zoom_level)
    sy = int((ry - pan_y) * zoom_level)
    return sx, sy

def to_real_coords(sx, sy):
    """แปลงพิกัดบนหน้าจอที่ซูม -> พิกัดจริง (1280x720)"""
    rx = int(sx / zoom_level + pan_x)
    ry = int(sy / zoom_level + pan_y)
    rx = max(0, min(W - 1, rx))
    ry = max(0, min(H - 1, ry))
    return rx, ry

def render_view():
    view_w = int(W / zoom_level)
    view_h = int(H / zoom_level)
    
    # จำกัดขอบเขตการแพนไม่ให้หลุดภาพ
    clamped_x = int(np.clip(pan_x, 0, W - view_w))
    clamped_y = int(np.clip(pan_y, 0, H - view_h))
    
    crop = base_frame[clamped_y:clamped_y + view_h, clamped_x:clamped_x + view_w]
    view_img = cv2.resize(crop, (W, H), interpolation=cv2.INTER_LINEAR)
    
    # วาดจุดและเส้นที่บันทึกไว้
    for i, pt in enumerate(points_real):
        sx, sy = to_screen_coords(pt[0], pt[1])
        cv2.circle(view_img, (sx, sy), 4, (0, 0, 255), -1)
        
        # ลากเส้นเชื่อมในแต่ละช่อง (4 จุดต่อช่อง)
        if i % 4 != 0:
            prev_sx, prev_sy = to_screen_coords(points_real[i - 1][0], points_real[i - 1][1])
            cv2.line(view_img, (prev_sx, prev_sy), (sx, sy), (0, 255, 0), 2)
            
        if (i + 1) % 4 == 0:
            first_sx, first_sy = to_screen_coords(points_real[i - 3][0], points_real[i - 3][1])
            cv2.line(view_img, (sx, sy), (first_sx, first_sy), (0, 255, 0), 2)
            slot_id = (i + 1) // 4
            cv2.putText(view_img, f"S{slot_id:02d}", (first_sx, first_sy - 8),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 100, 0), 2)
            
    # HUD บอกสถานะการซูม
    hud_text = f"Zoom: {zoom_level:.1f}x | Slots: {len(points_real)//4}/10 | Scroll: Zoom, Right-Drag: Pan"
    cv2.rectangle(view_img, (10, H - 40), (600, H - 10), (20, 20, 20), -1)
    cv2.putText(view_img, hud_text, (20, H - 20), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 255), 1)
    
    return view_img

def mouse_callback(event, x, y, flags, param):
    global zoom_level, pan_x, pan_y, is_dragging, drag_start, points_real, all_slots
    
    # 1. คลิกซ้าย: มาร์กจุด
    if event == cv2.EVENT_LBUTTONDOWN:
        rx, ry = to_real_coords(x, y)
        points_real.append([rx, ry])
        
        if len(points_real) % 4 == 0:
            slot_id = len(points_real) // 4
            slot_poly = points_real[-4:]
            all_slots.append(slot_poly)
            print(f"    np.array({slot_poly}, np.int32),  # SLOT {slot_id:02d}")
            
    # 2. คลิกขวาค้างเพื่อ Pan ภาพ
    elif event == cv2.EVENT_RBUTTONDOWN:
        is_dragging = True
        drag_start = (x, y)
        
    elif event == cv2.EVENT_MOUSEMOVE:
        if is_dragging:
            dx = int((drag_start[0] - x) / zoom_level)
            dy = int((drag_start[1] - y) / zoom_level)
            pan_x = np.clip(pan_x + dx, 0, W - int(W / zoom_level))
            pan_y = np.clip(pan_y + dy, 0, H - int(H / zoom_level))
            drag_start = (x, y)
            
    elif event == cv2.EVENT_RBUTTONUP:
        is_dragging = False
        
    # 3. หมุนล้อเมาส์เพื่อ Zoom In / Out
    elif event == cv2.EVENT_MOUSEWHEEL:
        old_zoom = zoom_level
        if flags > 0:
            zoom_level = min(zoom_level * 1.25, 6.0)
        else:
            zoom_level = max(zoom_level / 1.25, 1.0)
            
        # คำนวณจุดกึ่งกลางการซูมตามตำแหน่งเมาส์
        mouse_rx = pan_x + (x / old_zoom)
        mouse_ry = pan_y + (y / old_zoom)
        pan_x = np.clip(mouse_rx - (x / zoom_level), 0, W - int(W / zoom_level))
        pan_y = np.clip(mouse_ry - (y / zoom_level), 0, H - int(H / zoom_level))

win_name = "Zoomable Parking Slot Picker"
cv2.namedWindow(win_name, cv2.WINDOW_NORMAL)
cv2.setMouseCallback(win_name, mouse_callback)

print("\n--- วิธีใช้งานแบบซูม ---")
print("• หมุนล้อเมาส์: ซูมเข้า / ซูมออก")
print("• คลิกขวาค้างแล้วลาก: เลื่อนหน้าจอ (Pan)")
print("• คลิกซ้าย 4 จุดต่อช่อง: [บนซ้าย -> บนขวา -> ล่างขวา -> ล่างซ้าย]")
print("• ปุ่ม R: รีเซ็ตจุดทั้งหมด")
print("• ปุ่ม Q: บันทึกและออก")
print("-------------------------\n")

while True:
    view = render_view()
    cv2.imshow(win_name, view)
    key = cv2.waitKey(15) & 0xFF
    if key == ord('q') or key == 27:
        break
    elif key == ord('r'):
        points_real = []
        all_slots = []
        print("↺ รีเซ็ตจุดทั้งหมดเรียบร้อย")

cv2.destroyAllWindows()

if all_slots:
    print("\n" + "="*50)
    print("SLOT_POLYGONS = [")
    for idx, poly in enumerate(all_slots):
        print(f"    np.array({poly}, np.int32),  # SLOT {idx+1:02d}")
    print("]")
    print("="*50)


--- วิธีใช้งานแบบซูม ---
• หมุนล้อเมาส์: ซูมเข้า / ซูมออก
• คลิกขวาค้างแล้วลาก: เลื่อนหน้าจอ (Pan)
• คลิกซ้าย 4 จุดต่อช่อง: [บนซ้าย -> บนขวา -> ล่างขวา -> ล่างซ้าย]
• ปุ่ม R: รีเซ็ตจุดทั้งหมด
• ปุ่ม Q: บันทึกและออก
-------------------------

    np.array([[230, 335], [118, 349], [166, 356], [289, 339]], np.int32),  # SLOT 01

SLOT_POLYGONS = [
    np.array([[230, 335], [118, 349], [166, 356], [289, 339]], np.int32),  # SLOT 01
]
